# Confluence REST API walkthrough

A runnable tour of the Confluence Cloud REST API using only the Python standard library (no `requests`, no `atlassian-python-api`). Each section explains one part of the API, then calls it.

| Section | API surface | Writes? |
|---|---|---|
| 1. Authentication | Basic auth with an API token; classic vs scoped base URL | no |
| 2. Request helper | JSON, errors, retries, `Retry-After` | no |
| 3. Spaces | `GET /wiki/api/v2/spaces` | no |
| 4. Pages and pagination | `GET /wiki/api/v2/pages`, cursor `_links.next` | no |
| 5. Storage format | XHTML body, macros, `contentbody/convert` (v1) | no |
| 6. Create or update a page | `POST` / `PUT /wiki/api/v2/pages`, version numbers | yes |
| 7. Attachments | `PUT /wiki/rest/api/content/{id}/child/attachment` (v1, multipart) | yes |
| 8. Labels and content properties | v1 labels, v2 properties | yes |
| 9. Read back | `GET /wiki/api/v2/pages/{id}?body-format=storage` | no |
| 10. Clean up | `DELETE /wiki/api/v2/pages/{id}` | yes |

Every write is gated by `WRITE`. With the default `WRITE = False` the notebook only reads and prints what it would send.

Notes: [[confluence-automation-api]], [[confluence-macro-cheatsheet]], [[confluence-run-page-template]].

## 1. Authentication

SAML SSO only protects browser login. REST calls skip the identity provider and authenticate with an **API token** created at `id.atlassian.com → Security → API tokens`.

| Token kind | Base URL | How to send it |
|---|---|---|
| Classic API token | `https://<site>.atlassian.net` | `Authorization: Basic base64(email:token)` |
| Scoped API token | `https://api.atlassian.com/ex/confluence/<cloudId>` | same Basic header |
| Data Center PAT | `https://<host>` (no v2 API) | `Authorization: Bearer <pat>` |

`cloudId` comes from `https://<site>.atlassian.net/_edge/tenant_info`. If token creation is disabled for your account, an org admin has blocked API tokens for managed accounts; ask for an exception or a service account.

All paths below start with `/wiki/...` and are appended to `ROOT`. `CONFLUENCE_BASE_URL` overrides `ROOT` (proxies, local mocks).

Required environment variables: `CONFLUENCE_EMAIL`, `DTRACK_CONFLUENCE_TOKEN`. Optional: `CONFLUENCE_SITE`, `CONFLUENCE_TOKEN_KIND`, `CONFLUENCE_CLOUD_ID`, `CONFLUENCE_SPACE`, `CONFLUENCE_PARENT`, `CONFLUENCE_ATTACHMENT`, `CONFLUENCE_WRITE=1`.

In [ ]:
import os

SITE = os.environ.get("CONFLUENCE_SITE", "<site>")
EMAIL = os.environ.get("CONFLUENCE_EMAIL")
TOKEN = os.environ.get("DTRACK_CONFLUENCE_TOKEN")
TOKEN_KIND = os.environ.get("CONFLUENCE_TOKEN_KIND", "classic")
CLOUD_ID = os.environ.get("CONFLUENCE_CLOUD_ID")
BASE_URL = os.environ.get("CONFLUENCE_BASE_URL")
SPACE_KEY = os.environ.get("CONFLUENCE_SPACE", "DT")
PARENT_TITLE = os.environ.get("CONFLUENCE_PARENT")
ATTACHMENT = os.environ.get("CONFLUENCE_ATTACHMENT")
PAGE_TITLE = "dtrack · rest api walkthrough"
LABELS = ["dtrack", "dtrack-api-test"]
WRITE = os.environ.get("CONFLUENCE_WRITE") == "1"
DELETE_TEST_PAGE = False

In [ ]:
import base64

if not EMAIL or not TOKEN:
    raise ValueError("Set CONFLUENCE_EMAIL and DTRACK_CONFLUENCE_TOKEN")
if BASE_URL:
    ROOT = BASE_URL.rstrip("/")
elif TOKEN_KIND == "scoped":
    if not CLOUD_ID:
        raise ValueError("Scoped tokens need CONFLUENCE_CLOUD_ID from https://<site>.atlassian.net/_edge/tenant_info")
    ROOT = f"https://api.atlassian.com/ex/confluence/{CLOUD_ID}"
elif TOKEN_KIND == "classic":
    ROOT = f"https://{SITE}.atlassian.net"
else:
    raise ValueError("CONFLUENCE_TOKEN_KIND must be classic or scoped")

AUTH = "Basic " + base64.b64encode(f"{EMAIL}:{TOKEN}".encode()).decode()
print("ROOT =", ROOT, "| WRITE =", WRITE)

## 2. Request helper

One function wraps `urllib.request`:

- JSON in, JSON out; `Accept: application/json` on every call.
- `429` (rate limit) and `5xx` are retried, honouring `Retry-After` when present, otherwise exponential backoff.
- Any other error raises `ConfluenceError` with the status and the first 500 characters of the response body, which usually names the bad field.

| Status | Typical cause |
|---|---|
| 400 | Malformed storage XHTML or a missing field |
| 401 | Wrong email/token, or API tokens blocked by org policy |
| 403 | Token valid, no permission on the space or page |
| 404 | Wrong id, page not visible, or `/api/v2` on Data Center |
| 409 | Version conflict or duplicate title |
| 413 | Attachment over the site limit |

In [ ]:
import json
import time
import urllib.error
import urllib.parse
import urllib.request


# >>> HTTP error raised by call() with status and response snippet <<< #
class ConfluenceError(RuntimeError):
    def __init__(self, status, method, url, text):
        super().__init__(f"{status} {method} {url}: {text[:500]}")
        self.status = status


# >>> send one REST request, retrying 429 and 5xx <<< #
def call(method, path, *, params=None, body=None, data=None, headers=None, retries=5):
    url = path if path.startswith("http") else ROOT + path
    if params:
        url += ("&" if "?" in url else "?") + urllib.parse.urlencode(params)
    head = {"Authorization": AUTH, "Accept": "application/json"}
    if body is not None:
        data = json.dumps(body).encode()
        head["Content-Type"] = "application/json"
    head.update(headers or {})
    for attempt in range(retries):
        request = urllib.request.Request(url, data=data, method=method, headers=head)
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                raw = response.read()
                return json.loads(raw) if raw else {}
        except urllib.error.HTTPError as err:
            text = err.read().decode(errors="replace")
            if err.code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                wait = err.headers.get("Retry-After")
                time.sleep(float(wait) if wait and wait.isdigit() else 2 ** attempt)
                continue
            raise ConfluenceError(err.code, method, url, text) from None

## 3. Spaces

v2 endpoints take the numeric `spaceId`, not the space key, so the first call resolves the key once.

`GET /wiki/api/v2/spaces?keys=DT` → `results[0]` has `id`, `key`, `name`, `homepageId`.

This is also the cheapest **smoke test**: a `200` here proves the token, the base URL and space visibility in one call.

In [ ]:
spaces = call("GET", "/wiki/api/v2/spaces", params={"keys": SPACE_KEY})["results"]
if not spaces:
    raise ValueError(f"Space {SPACE_KEY} not found or not visible to this token")
SPACE = spaces[0]
print(f"space {SPACE['key']} id={SPACE['id']} name={SPACE['name']!r} homepage={SPACE.get('homepageId')}")

## 4. Pages and pagination

- Titles are unique per space, so `GET /wiki/api/v2/pages?space-id=&title=` returns zero or one current page. This is the basis of idempotent publishing: find first, then create or update.
- List responses are cursor-paginated. Follow `_links.next` (a path starting with `/wiki/...`) until it disappears; never build cursors yourself.
- `limit` caps the page size (v2 allows up to 250).

The parent defaults to the space homepage when `CONFLUENCE_PARENT` is unset.

In [ ]:
from itertools import islice


# >>> yield every result across cursor-paginated v2 responses <<< #
def paginate(path, params=None):
    page = call("GET", path, params=params)
    while True:
        yield from page.get("results", [])
        link = page.get("_links", {}).get("next")
        if not link:
            return
        page = call("GET", link)


# >>> return the current page with this exact title in SPACE, or None <<< #
def find_page(title):
    hits = call("GET", "/wiki/api/v2/pages", params={"space-id": SPACE["id"], "title": title, "status": "current"})["results"]
    return hits[0] if hits else None


if PARENT_TITLE:
    parent = find_page(PARENT_TITLE)
    if parent is None:
        raise ValueError(f"Parent page {PARENT_TITLE!r} not found in {SPACE_KEY}")
    PARENT_ID = parent["id"]
else:
    PARENT_ID = SPACE["homepageId"]

print("parent id:", PARENT_ID)
for child in islice(paginate(f"/wiki/api/v2/pages/{PARENT_ID}/children", {"limit": 25}), 10):
    print(f"  child {child['id']}: {child['title']}")
existing = find_page(PAGE_TITLE)
print("test page:", f"exists id={existing['id']} v{existing['version']['number']}" if existing else "not created yet")

## 5. Storage format

Page bodies are **storage format**: XHTML plus Confluence elements.

- Macros: `<ac:structured-macro ac:name="status">` with `<ac:parameter>` children.
- References: `<ri:page>`, `<ri:attachment>`, `<ri:user>` inside `<ac:link>`.
- Every dynamic string must be XML-escaped; every tag closed.

`POST /wiki/rest/api/contentbody/convert/view` (v1) renders storage to HTML without saving anything, so it doubles as a validator: malformed XHTML returns `400` before any page is touched.

In [ ]:
from datetime import datetime
from pathlib import Path
from xml.sax.saxutils import escape, quoteattr


# >>> render a status lozenge macro <<< #
def status(title, colour):
    return (
        '<ac:structured-macro ac:name="status">'
        f'<ac:parameter ac:name="colour">{colour}</ac:parameter>'
        f'<ac:parameter ac:name="title">{escape(title)}</ac:parameter>'
        "</ac:structured-macro>"
    )


# >>> render a link to an attachment on the same page <<< #
def attachment_link(filename, text):
    return f"<ac:link><ri:attachment ri:filename={quoteattr(filename)}/><ac:plain-text-link-body><![CDATA[{text}]]></ac:plain-text-link-body></ac:link>"


rows = [
    ("orders", "MATCH", "Green", "DIFF", "Yellow"),
    ("users", "MISSING", "Red", "SKIPPED", "Grey"),
    ("events & logs", "MATCH", "Green", "PASS", "Green"),
]
generated = datetime.now().astimezone().strftime("%Y-%m-%d %H:%M %Z")
attachment = Path(ATTACHMENT) if ATTACHMENT else None
table_rows = "".join(f"<tr><td>{escape(pair)}</td><td>{status(row, rc)}</td><td>{status(col, cc)}</td></tr>" for pair, row, rc, col, cc in rows)

XHTML = (
    '<ac:structured-macro ac:name="info"><ac:rich-text-body>'
    f"<p>Written by the REST API walkthrough notebook at {escape(generated)}.</p>"
    "</ac:rich-text-body></ac:structured-macro>"
    "<table><thead><tr><th>Pair</th><th>Row check</th><th>Col check</th></tr></thead>"
    f"<tbody>{table_rows}</tbody></table>"
    + (f"<p>{attachment_link(attachment.name, 'Attachment')}</p>" if attachment else "")
)

try:
    view = call("POST", "/wiki/rest/api/contentbody/convert/view", body={"value": XHTML, "representation": "storage"})
    print(f"storage XHTML valid: {len(XHTML)} chars -> {len(view.get('value', ''))} chars of HTML")
except ConfluenceError as err:
    if err.status == 400:
        raise
    print("convert endpoint unavailable, skipping validation:", err)

## 6. Create or update a page

| Operation | Call | Body |
|---|---|---|
| Create | `POST /wiki/api/v2/pages` | `spaceId`, `parentId`, `status: current`, `title`, `body{representation, value}` |
| Update | `PUT /wiki/api/v2/pages/{id}` | same, plus `id` and `version{number: current + 1, message}` |

Rules:

- Updates must send exactly `current + 1`; anything else returns `409`. Re-read and retry if someone edited the page meanwhile.
- Every successful update creates a page version and may notify watchers, so skip the call when nothing changed (store a hash in a content property, section 8).
- Moving a page means changing `parentId`; renaming means changing `title`. Both break `ri:content-title` links elsewhere.

In [ ]:
# >>> create the page if missing, otherwise bump its version with a new body <<< #
def upsert_page(title, xhtml, message):
    page = find_page(title)
    body = {"representation": "storage", "value": xhtml}
    if page is None:
        return call("POST", "/wiki/api/v2/pages", body={"spaceId": SPACE["id"], "parentId": PARENT_ID, "status": "current", "title": title, "body": body})
    version = {"number": page["version"]["number"] + 1, "message": message}
    return call("PUT", f"/wiki/api/v2/pages/{page['id']}", body={"id": page["id"], "status": "current", "title": title, "body": body, "version": version})


if WRITE:
    PAGE = upsert_page(PAGE_TITLE, XHTML, f"rest api walkthrough {generated}")
    print(f"page {PAGE['id']} now at version {PAGE['version']['number']}")
else:
    PAGE = existing
    print(f"dry run: would {'update' if existing else 'create'} {PAGE_TITLE!r} under {PARENT_ID} with {len(XHTML)} chars")

## 7. Attachments

v2 can list attachments but cannot upload them, so uploads use v1:

- `PUT /wiki/rest/api/content/{pageId}/child/attachment` creates the file or adds a new version if the filename already exists (`POST` on the same path fails on duplicates).
- Body is `multipart/form-data` with a `file` part; optional `comment` and `minorEdit` parts.
- Header `X-Atlassian-Token: nocheck` is mandatory (XSRF guard), otherwise `403`.
- Filenames are unique per page; link to them with `<ri:attachment ri:filename="..."/>`.

`GET /wiki/api/v2/pages/{id}/attachments` lists what is already there (`title`, `fileSize`, `mediaType`, `version`).

In [ ]:
import mimetypes
import uuid


# >>> encode form fields plus one file as multipart/form-data <<< #
def multipart(fields, file_field, path):
    boundary = uuid.uuid4().hex
    parts = [f'--{boundary}\r\nContent-Disposition: form-data; name="{name}"\r\n\r\n{value}\r\n'.encode() for name, value in fields.items()]
    kind = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    parts.append(f'--{boundary}\r\nContent-Disposition: form-data; name="{file_field}"; filename="{path.name}"\r\nContent-Type: {kind}\r\n\r\n'.encode())
    parts.append(path.read_bytes())
    parts.append(f"\r\n--{boundary}--\r\n".encode())
    return b"".join(parts), f"multipart/form-data; boundary={boundary}"


# >>> upload a file to a page, creating a new version if the name exists <<< #
def put_attachment(page_id, path, comment):
    data, kind = multipart({"comment": comment, "minorEdit": "true"}, "file", path)
    return call("PUT", f"/wiki/rest/api/content/{page_id}/child/attachment", data=data, headers={"Content-Type": kind, "X-Atlassian-Token": "nocheck"})


if attachment is None:
    print("no CONFLUENCE_ATTACHMENT set, skipping upload")
elif not WRITE or PAGE is None:
    print(f"dry run: would upload {attachment.name} ({attachment.stat().st_size} bytes)")
else:
    uploaded = put_attachment(PAGE["id"], attachment, f"rest api walkthrough {generated}")
    print("uploaded:", [item["title"] for item in uploaded.get("results", [uploaded])])

if PAGE is not None:
    for item in paginate(f"/wiki/api/v2/pages/{PAGE['id']}/attachments", {"limit": 50}):
        print(f"  attachment {item['title']} {item.get('fileSize')} bytes v{item.get('version', {}).get('number')}")

## 8. Labels and content properties

**Labels** (v1 write, v2 read) drive CQL, Content by Label and the Page Properties Report:

- `POST /wiki/rest/api/content/{id}/label` with `[{"prefix": "global", "name": "dtrack-run"}]`; lowercase, no spaces; adding an existing label is a no-op.
- `GET /wiki/api/v2/pages/{id}/labels` to read.

**Content properties** (v2) are JSON key/value pairs on the page, invisible to readers. Use them for machine state such as a source hash, so the publisher can skip unchanged runs. They are unrelated to the Page Properties *macro*.

- `GET /wiki/api/v2/pages/{id}/properties?key=...`
- `POST /wiki/api/v2/pages/{id}/properties` with `{"key", "value"}`
- `PUT /wiki/api/v2/pages/{id}/properties/{propertyId}` with `{"key", "value", "version": {"number": n + 1}}`

In [ ]:
import hashlib


# >>> create or update one content property on a page <<< #
def set_property(page_id, key, value):
    found = call("GET", f"/wiki/api/v2/pages/{page_id}/properties", params={"key": key})["results"]
    if not found:
        return call("POST", f"/wiki/api/v2/pages/{page_id}/properties", body={"key": key, "value": value})
    prop = found[0]
    return call("PUT", f"/wiki/api/v2/pages/{page_id}/properties/{prop['id']}", body={"key": key, "value": value, "version": {"number": prop["version"]["number"] + 1}})


body_hash = hashlib.sha256(XHTML.encode()).hexdigest()
if WRITE and PAGE is not None:
    call("POST", f"/wiki/rest/api/content/{PAGE['id']}/label", body=[{"prefix": "global", "name": name} for name in LABELS])
    set_property(PAGE["id"], "dtrack-body-sha256", body_hash)
    labels = [item["name"] for item in paginate(f"/wiki/api/v2/pages/{PAGE['id']}/labels")]
    print("labels:", labels, "| body hash stored:", body_hash[:12])
else:
    print(f"dry run: would add labels {LABELS} and property dtrack-body-sha256={body_hash[:12]}")

## 9. Read back

`GET /wiki/api/v2/pages/{id}?body-format=storage` returns the stored body, `version`, and `_links` (`webui` is relative to `_links.base`).

Confluence normalises stored XHTML (attribute order, whitespace, macro ids), so do not diff it against what you sent; compare the hash property instead.

In [ ]:
if PAGE is None:
    print("nothing to read back yet; rerun with CONFLUENCE_WRITE=1")
else:
    current = call("GET", f"/wiki/api/v2/pages/{PAGE['id']}", params={"body-format": "storage"})
    links = current.get("_links", {})
    print(f"{current['title']} v{current['version']['number']}: {links.get('base', ROOT + '/wiki')}{links.get('webui', '')}")
    print("stored body:", len(current["body"]["storage"]["value"]), "chars")

## 10. Clean up

`DELETE /wiki/api/v2/pages/{id}` moves the page to the space trash (recoverable by a space admin). Adding `?purge=true` on an already trashed page deletes it permanently.

Set `DELETE_TEST_PAGE = True` in the config cell to remove the test page.

In [ ]:
if DELETE_TEST_PAGE and WRITE and PAGE is not None:
    call("DELETE", f"/wiki/api/v2/pages/{PAGE['id']}")
    print(f"moved {PAGE['id']} to trash")
else:
    print("test page kept")